# Meter Maintenance Notebook

Use the main notebook for the normal pipeline run.

- inspect one meter or a few meters closely
- compare raw data to the current broken meter source entry
- review the raw readings captured inside remove/broken windows
- decide whether a meter is still broken, repaired, misclassified, or needs a date change
- keep a small update log
- create a reviewed copy of the broken meter source file
- optionally write confirmed changes back to the source file

### Files explanation

- `running_list_broken_meters.xlsx` = **source of truth** broken meter file
- `removed_special_meter_data.csv` = **raw readings captured** inside remove/broken windows
- `special_meter_candidates.xlsx` = current review candidate workbook
- `special_meters_corrections_master_sheet.xlsx` = generated master corrections workbook


In [3]:
import os
import importlib
import numpy as np
import pandas as pd
import data_clean_TEST as dc
importlib.reload(dc)

<module 'data_clean_TEST' from '/Users/cassiehuber/Documents/GitHub/harvest_kwh_prep/notebooks/data_clean_TEST.py'>

### 1. Parameters

In [ ]:
############ CHANGE PARAMETERS AS NEEDED #############

# one meter or a short list of meters to inspect
meters_to_review = [
    # "pbrc_main_b",
]

# optional zoom window for plots
zoom_start = None   # e.g. "2025-08-01 00:00:00"
zoom_end = None     # e.g. "2025-10-01 00:00:00"

# whether to save changes back to the source broken meter file
write_changes_to_source_file = False

######################################################

In [ ]:
# Paths
input_dir = "../data/extracts/"
output_dir = "../data/outputs/"
plot_dir = "../data/outputs/plots/"
maintenance_dir = "../data/outputs/maintenance/"

os.makedirs(output_dir, exist_ok=True)
os.makedirs(plot_dir, exist_ok=True)
os.makedirs(maintenance_dir, exist_ok=True)

# TODO: change to server paths if running on server, make be all of its meter data or a subset
# Main raw meter data for this run
var_file = input_dir + "harvest_interval_kwh_" + "250723-251017.csv"

# TODO: FIX

# Main pipeline source/review files
meter_issues_candidates_file = input_dir + "special_meter_candidates.xlsx"
meter_issues_file = input_dir + "special_meters.xlsx"
broken_meters_file = input_dir + "running_list_broken_meters.xlsx"

# Main pipeline generated files
meter_corrections_file = output_dir + "special_meters_corrections_master_sheet.xlsx"
removed_special_meter_data_file = output_dir + "removed_special_meter_data.csv"

# Maintenance files
broken_meter_update_log_file = maintenance_dir + "broken_meter_update_log.csv"
broken_meter_reviewed_copy_file = maintenance_dir + "running_list_broken_meters_reviewed_copy.xlsx"
maintenance_plot_file = plot_dir + "maintenance_review_selected_meters.pdf"


section ideas:

- load one meter or selected meters
- plot raw data around chosen dates
- compare against current broken list entry
- decide update: broken / repaired / not actually broken
- save changes back to the source file / or database


In [ ]:
def _empty_removed_df():
    return pd.DataFrame(columns=[
        "datetime",
        "meter_name",
        "meter_reading",
        "solution",
        "correction_start",
        "correction_end",
        "issue_type/status",
        "description",
    ])

### Load raw data and review/source files

In [ ]:
# Load raw interval data
#TODO: change to load from server
raw_df = pd.read_csv(var_file)
raw_df["datetime"] = pd.to_datetime(raw_df["datetime"], errors="coerce")
raw_df = raw_df.dropna(subset=["datetime"]).copy()

required_raw_cols = {"datetime", "meter_name", "meter_reading"}
missing_raw_cols = required_raw_cols.difference(set(raw_df.columns))
if missing_raw_cols:
    raise ValueError(f"Raw interval file is missing required columns: {sorted(missing_raw_cols)}")

# Pivot to wide meter dataframe and fill missing timestamps
pivoted_df = raw_df.pivot(index="datetime", columns="meter_name", values="meter_reading").reset_index()
full_df = dc.fill_missing_timestamps(pivoted_df, "15min")
data = full_df.set_index("datetime").sort_index()

# Load broken meter source file
broken_source_df = dc.load_broken_meter_workbook(broken_meters_file)

# Load candidate workbook
candidate_df = dc._load_existing_candidates(meter_issues_candidates_file)

# Load manual special meters source
manual_special_df = dc._load_base_master_corrections(meter_issues_file)

# Load generated master corrections workbook if present
if os.path.exists(meter_corrections_file):
    master_corrections_df = pd.read_excel(meter_corrections_file)
else:
    master_corrections_df = pd.DataFrame(columns=[
        "meter_name", "solution", "start_datetime", "end_datetime", "issue_type/status", "description"
    ])

# Load removed raw data captured inside remove/broken windows
if os.path.exists(removed_special_meter_data_file):
    removed_df = pd.read_csv(removed_special_meter_data_file)
    for col in ["datetime", "correction_start", "correction_end"]:
        if col in removed_df.columns:
            removed_df[col] = pd.to_datetime(removed_df[col], errors="coerce")
else:
    removed_df = _empty_removed_df()

print("raw_df shape:", raw_df.shape)
print("data shape:", data.shape)
print("broken_source_df rows:", len(broken_source_df))
print("removed_df rows:", len(removed_df))


In [ ]:
if meters_to_review:
    selected_broken = 

### Meter review plot(s)

In [ ]:
meter_name_norm = str(meter_name).strip().lower()

if meter_name_norm not in [str(col).strip().lower() for col in data.columns]:
    print(f"{meter_name} not found in data columns.")

# map normalized meter name back to actual data column name
data_col = None
for col in data.columns:
    if str(col).strip().lower() == meter_name_norm:
        data_col = col
        break

meter_series = data[data_col].copy()

meter_broken = broken_source_df[broken_source_df["meter_name"] == meter_name_norm].copy()
meter_candidates = candidate_df[candidate_df["meter_name"] == meter_name_norm].copy()
meter_removed = removed_df[removed_df["meter_name"].astype(str).str.strip().str.lower() == meter_name_norm].copy()

plot_start = pd.to_datetime(meter_series.index.min())
plot_end = pd.to_datetime(meter_series.index.max())